# C4: automated Mistral adapter updates
Run in your **existing GPU Colab session**. This worker polls your app, trains actual QLoRA adapter weights, compares unmodified baseline/candidate outputs and reports them to the server. The server decides whether evaluation passed. No dataset is bundled.

**Prerequisites:** a public HTTPS app/API origin (localhost cannot work from Colab), separate admin training consent, at least 20 training and 5 held-out reviewed campaign briefs, and a temporary worker token from Email Agent → Learning center.

The app's `OLLAMA_HOST` must point to Ollama in this same runtime to use trained candidates. Keep your existing inference tunnel running; this notebook does not open a public tunnel. Training temporarily pauses AI generation on this shared GPU. Manual drafting remains available.

The runtime must use the project's pinned Ollama **0.12.3**, the matching Mistral v0.3 base and the adapter used to build your current default model. This notebook does not replace a running Ollama installation. See the included guide for compatibility and Colab limitations.

In [ ]:
# 1. Upload colab-ft-worker.zip from Learning center. The kit contains code only.
from google.colab import files
from pathlib import Path
import zipfile, io
KIT = Path('/content/c4-qlora-worker')
KIT.mkdir(exist_ok=True)
uploaded = files.upload()
if len(uploaded) != 1: raise ValueError('Upload exactly the Colab worker setup ZIP.')
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    allowed = {'colab_worker.py', 'requirements-colab-worker.txt', 'c4_continuous_qlora_colab.ipynb', 'README.md'}
    for item in archive.infolist():
        if item.filename not in allowed or item.file_size > 300000: raise ValueError('Unexpected setup kit entry.')
        if (item.external_attr >> 16) & 0o170000 == 0o120000: raise ValueError('Symbolic links are not allowed.')
    archive.extractall(KIT)
del uploaded
print('Worker code loaded. No tenant data or credentials were included.')

In [ ]:
# 2. Install worker libraries without replacing Colab's CUDA torch.
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(KIT / 'requirements-colab-worker.txt')])
from google.colab import drive
drive.mount('/content/drive')
print('If these libraries were already imported in this session, restart the runtime and rerun setup before training.')

In [ ]:
# 3. Configure your public app address. Do not use the model tunnel URL here.
import os, getpass, subprocess, sys
API_URL = input('Public HTTPS app/API origin (no path): ').strip()
INITIAL_ADAPTER = input('Drive path of the adapter used by your current default model [/content/drive/MyDrive/c4_finetune/c4-email-lora]: ').strip() or '/content/drive/MyDrive/c4_finetune/c4-email-lora'
try:
    from google.colab import userdata
    token = userdata.get('C4_FT_WORKER_TOKEN')
except Exception:
    token = None
os.environ['C4_FT_WORKER_TOKEN'] = token or getpass.getpass('Temporary Colab worker token (hidden): ')
del token
WORKER = [sys.executable, str(KIT / 'colab_worker.py'), '--api-url', API_URL, '--initial-adapter', INITIAL_ADAPTER]
subprocess.check_call(WORKER + ['--check'])
# This checks local CUDA + pinned Ollama. It does not claim a job or train.
# Before continuing, confirm BOTH the default model and mistral:7b-instruct-v0.3-q4_K_M
# are installed in local Ollama (ollama list). Candidate registration uses that exact base.

In [ ]:
# 4. Keep this cell running while using the connected Colab session.
# Enable auto-queue in Learning center, or queue an eligible job there manually.
# The app's default is manual activation after evaluation, with optional automatic activation.
# Interrupt the cell to stop polling. A disconnected job loses its lease within 5 minutes.
try:
    subprocess.check_call(WORKER)
finally:
    os.environ.pop('C4_FT_WORKER_TOKEN', None)
    print('Worker stopped. Re-enter a valid token in step 3 before restarting.')